# 22 · Thiên lệch "có" của SH — chẩn đoán logit + sửa bằng ngưỡng lúc suy luận (W13, R-YB)

**Vấn đề (W12, POPE-vi):** adapter dùng chung **SH** (adapter của prototype) trả lời "có" ở 77–89% câu "Trong ảnh có X không?". So với zero-shot B1-ocr, ΔF1 là −5,7 / −9,2 / −9,4 điểm. Nếu đưa nguyên SH vào prototype, người dùng hỏi có/không gần như luôn nhận "có".

**Câu hỏi chẩn đoán:** đây là **lệch ngưỡng** (điểm "có" bị đẩy lên, nhưng câu có-thật vẫn xếp trên câu không-có → dời ngưỡng là sửa được) hay **mất khả năng phân biệt** (→ phải train lại, ≈ 7 h L4)?

**Cách đo:** một lượt forward, **không sinh chữ**, cùng input với greedy. Từ đó lấy log-xác suất token đầu:
- `margin = log P(có) − log P(không)`; quyết định "có" ⇔ `margin > τ`. Với τ = 0 thì gần như trùng greedy hiện tại.
- **AUC** của margin (chỉ mô tả) cho biết model còn phân biệt được hay không.
- **τ chọn CHỈ trên tập calibration**: ảnh ViVQA test chưa thấy lúc train và **không** thuộc 500 ảnh POPE-vi. Sau đó áp τ lên POPE-vi **một lần**.

**Luật R-YB** (`docs/W13/README.md`, commit TRƯỚC khi chạy) — ĐẠT khi đủ cả:
1. Hợp lệ: quyết định logit ở τ = 0 trùng greedy W12 ≥ 98% (SH và B1-ocr).
2. ΔF1 (SH với τ*) so với B1-ocr greedy: cận dưới CI95 > −2 ở cả 3 cách chọn.
3. yes-ratio của SH với τ* ≤ 60% ở cả 3 cách chọn.

KHÔNG ĐẠT → Fix B (train lại SH có câu có/không cân bằng), notebook riêng.

| Phần | Runtime | Việc | Thời gian |
|---|---|---|---|
| 1–3 | L4 | thiết lập, kiểm adapter + OCR, sinh 2 file câu hỏi | ≈ 10 phút |
| 4 | L4 | 3 cấu hình (SH, B1-ocr, B3-full) × 3 tập câu hỏi = 9 lượt chấm logit | ≈ 2–3 giờ (≈ 4 CU) |
| 5 | L4 | phân tích + phán quyết, tự trả máy | ≈ 1 phút |

## 1. Thiết lập môi trường

Mount Drive, lấy code mới nhất từ GitHub, khai báo đường dẫn. Notebook này chạy **toàn bộ trên L4**. Phần sinh câu hỏi chỉ mất vài phút, nên không tách sang CPU cho đỡ phải đổi runtime.

*Kỳ vọng: commit mới nhất (có notebook 22), `Runtime: NVIDIA L4`, `✅ Thiết lập xong`.*

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, hashlib, subprocess
if not os.path.exists('/content/ViVQA-VLM/.git'):
    !cd /content && git clone -q https://github.com/PLHGNAOH/ViVQA-VLM.git
%cd /content/ViVQA-VLM
!git pull -q origin main
!git log -1 --oneline
sys.path.insert(0, '/content/ViVQA-VLM')
assert os.path.exists('src/eval/yesno_calib.py'), 'Code cũ -> git pull chưa lấy được commit W13 (yesno_calib.py)'

DRIVE_ROOT = '/content/drive/MyDrive/ViVQA-VLM'
EXP_ROOT = f'{DRIVE_ROOT}/experiments'
VIVQA_DIR = f'{DRIVE_ROOT}/data/vivqa'
POPE_REPO = 'experiments/W12_pope_vi_questions/pope_vi_questions_v1.json'   # bộ câu hỏi W12 đã chốt
POPE_SHA = '53d04eb23ad41ecee372d6dee01201578de75d992d33dd2c5b94d4aa328928fd'
Q_DIR = f'{EXP_ROOT}/W13_yesno_questions'                                    # 2 file câu hỏi mới (calib, t2)
OUT = f'{EXP_ROOT}/W13_yesno_calib_L4_seed42'
CHUNK = 500

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''):
            h.update(b)
    return h.hexdigest()

def run_cmd(cmd):
    # Chạy lệnh, in log trực tiếp; lỗi -> DỪNG notebook
    print('$', cmd)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='')
    if p.wait() != 0:
        raise RuntimeError(f'Lệnh lỗi (mã {p.returncode}) -> xem log ở trên, gửi Claude')

GPU = subprocess.run('nvidia-smi --query-gpu=name --format=csv,noheader', shell=True,
                     capture_output=True, text=True).stdout.strip() or 'CPU'
print('Runtime:', GPU)
print('✅ Thiết lập xong')

## 2. Kiểm tra trước khi tốn ≈ 2–3 giờ L4

Phần này **bắt buộc L4**, cùng GPU với các run greedy W12. Nếu đang ở runtime khác thì vào **Runtime → Change runtime type → L4 → Save**, rồi **Runtime → Run all**.

Các bước kiểm:
- **(a)** Cài thư viện, cùng pin `transformers==5.17.0` như notebook 21.
- **(b)** Cấu hình OCR đọc từ `ocr_selection.json` (W08), không gõ tay. OCR cache ViVQA phải trùng sha256 biên bản W11.
- **(c)** Adapter SH và B3-full trên Drive phải **trùng dấu vân tay trọng số** với run gốc (W10/W11).
- **(d)** File POPE-vi trong repo phải trùng sha256 đã chốt ở W12.
- **(e)** Selftest `yesno_calib` + `pope_vi`.

*Kỳ vọng: `GPU: NVIDIA L4`, `paddledet_vietocr | v1 | 600 | 0.0`, `OCR cache ViVQA = biên bản W11 ✅`, 2 dòng `adapter … = run gốc ✅`, `POPE-vi = bản W12 ✅`, `✅ selftest OK (2 script)`.*

In [ ]:
if 'L4' not in GPU:
    raise SystemExit('❌ Notebook 22 phải chạy trên L4 (cùng GPU với run greedy W12). '
                     'Runtime -> Change runtime type -> L4, rồi Run all.')
print('GPU:', GPU)
!pip install -q transformers==5.17.0 accelerate qwen-vl-utils bitsandbytes peft huggingface_hub pyyaml

from src.data.ocr import default_cache_path, load_ocr_cache
from src.eval.run_baseline import adapter_fingerprint
from src.eval.pope_vi import REF_EVAL

# (b) OCR: cấu hình đóng băng W08 + cache ViVQA đúng biên bản W11
sel = json.load(open('experiments/W08_ocr_select_dev300_L4_seed42/ocr_selection.json', encoding='utf-8'))['decision']
BACKEND, TPL, MAXC, MINC = sel['backend'], sel['template'], sel['max_chars'], sel['min_conf']
print(BACKEND, '|', TPL, '|', MAXC, '|', MINC)
OCR_ARGS = f'--mode ocr --ocr_backend {BACKEND} --ocr_template {TPL} --ocr_max_chars {MAXC} --ocr_min_conf {MINC}'
C_VV = default_cache_path(DRIVE_ROOT, 'vivqa', BACKEND)
MAN_VV = f'{EXP_ROOT}/W11_ocr_cache_vivqa/ocr_cache_manifest.json'
assert sha256_file(C_VV) == json.load(open(MAN_VV, encoding='utf-8'))['sha256'], 'OCR cache ViVQA KHÁC biên bản W11 -> dừng, báo Claude'
print('OCR cache ViVQA = biên bản W11 ✅')

# (c) adapter = đúng adapter của run gốc (so dấu vân tay trọng số, không so đường dẫn)
AD = {}
for k in ('SH', 'B3full'):
    ref = json.load(open(f'{EXP_ROOT}/{REF_EVAL[k]}/metrics.json', encoding='utf-8'))['meta']['adapter']
    fp = adapter_fingerprint(ref['path'])
    assert fp['weights_sha256'] == ref['weights_sha256'], f'{k}: adapter trên Drive KHÁC run gốc -> dừng, báo Claude'
    AD[k] = ref['path']
    print(f"adapter {k:6s} = run gốc ✅  ({fp['weights_mb']} MB, {fp['weights_sha256'][:12]}…)  {ref['path']}")

# (d) bộ câu hỏi POPE-vi W12
assert sha256_file(POPE_REPO) == POPE_SHA, 'POPE-vi trong repo KHÁC bản W12 -> dừng, báo Claude'
print('POPE-vi = bản W12 ✅')

# (e) selftest
for mod, must in {'src.eval.yesno_calib': 'yesno_calib selftest OK', 'src.eval.pope_vi': 'pope_vi selftest OK'}.items():
    o = subprocess.run(f'python -m {mod} --selftest', shell=True, capture_output=True, text=True)
    assert must in o.stdout, (mod, o.stdout[-800:], o.stderr[-800:])
print('✅ selftest OK (2 script)')

## 3. Sinh 2 file câu hỏi (một lần duy nhất)

1. **Calibration** `yesno_calib_v1.json`: cùng mẫu câu POPE-vi, trên ảnh ViVQA test **chưa thấy lúc train** và **không** thuộc 500 ảnh POPE-vi. Ảnh cần ≥ 1 lớp COCO, lấy tối đa 600 ảnh, seed 42. Số câu "có" = số câu "không" ở mỗi cách chọn. Tập này **chỉ dùng để chọn τ**, không dùng để đánh giá.
2. **POPE-vi mẫu câu 2** `pope_vi_questions_t2_v1.json`: cùng ảnh, cùng vật với POPE-vi, nhưng hỏi `Có {vật} trong ảnh không?` và **không có** chỉ dẫn "Trả lời có hoặc không". Mẫu này gần với câu người dùng thật gõ vào prototype. Chỉ mô tả.

Cần annotation COCO 2014 (≈ 250 MB) để biết vật nào có trong ảnh. File tải về **đĩa tạm** `/content`, không lên Drive. Nếu 2 file câu hỏi đã có trên Drive thì cell bỏ qua bước tải và **không sinh lại**.

Không cần điểm dừng commit như W12: file sinh **tất định** từ code + seed đã commit trước khi chạy. sha256 của file được ghi vào summary và Cursor chép file vào repo ở bước đóng.

*Kỳ vọng: `✔ yesno_calib_v1.json: … câu | sha256 …`, dòng `pool … ảnh -> chọn 600 | có … / không …`, `✔ pope_vi_questions_t2_v1.json: 4985 câu`, rồi `ảnh calib có OCR: …/… ✅`.*

In [ ]:
from src.eval.yesno_calib import QFILES
need_build = any(not os.path.exists(f'{Q_DIR}/{f}') for f in QFILES.values())
COCO_DIR = '/content/coco_ann'
if need_build:
    os.makedirs(COCO_DIR, exist_ok=True)
    if not os.path.exists('/content/annotations_trainval2014.zip'):
        !wget -q --show-progress -O /content/annotations_trainval2014.zip http://images.cocodataset.org/annotations/annotations_trainval2014.zip
    !unzip -j -o -q /content/annotations_trainval2014.zip annotations/instances_train2014.json annotations/instances_val2014.json -d {COCO_DIR}
run_cmd(f'python -m src.eval.yesno_calib --build --coco_ann_dir {COCO_DIR} --vivqa_dir {VIVQA_DIR} '
        f'--pope {POPE_REPO} --out_dir {Q_DIR}')
Q_FILES = {'calib': f"{Q_DIR}/{QFILES['calib']}", 'popevi': POPE_REPO, 'popevi_t2': f"{Q_DIR}/{QFILES['popevi_t2']}"}
for k, p in Q_FILES.items():
    print(f'{k:10s} {len(json.load(open(p, encoding="utf-8"))["data"]):5d} câu | sha256 {sha256_file(p)}')
cal = json.load(open(Q_FILES['calib'], encoding='utf-8'))['data']
need, have = {q['image'] for q in cal}, set(load_ocr_cache(C_VV)['images'])
print(f'ảnh calib có OCR: {len(need & have)}/{len(need)}', '✅' if need <= have else '')
assert need <= have, 'OCR cache thiếu ảnh calibration -> báo Claude'

## 4. Chấm logit: 3 cấu hình × 3 tập câu hỏi (≈ 2–3 giờ)

| Cấu hình | Vai trò |
|---|---|
| **SH** | đối tượng cần sửa (adapter prototype) |
| **B1-ocr** | mốc zero-shot cùng prompt (giống luật H4) |
| **B3-full** | adapter ViTextVQA chính thức, để so sánh (chỉ mô tả) |

Mỗi lượt chỉ có **một forward**, không sinh chữ, nên nhanh hơn greedy W12 (0,57 s/câu). Thứ tự chạy: SH trước, rồi B1-ocr, rồi B3-full.

Mỗi lô 500 câu được ghi lên Drive ngay khi xong. **Bị ngắt thì Run all lại**: lượt nào đã xong tự bỏ qua, lượt đang dở chạy tiếp từ lô cuối. Giữ máy không ngủ và tab mở.

*Kỳ vọng: mỗi lượt in `token có: [...] -> [...] | không: [...] -> [...]` (từ tiếng Việt và tiếng Anh nào là 1 token), sau đó là các dòng lô, cuối cùng `== xong … câu | … s/câu`. Mỗi lượt xong là một **điểm dừng an toàn**.*

In [ ]:
from src.eval.yesno_calib import CONFIGS, QSETS, logit_run_name
for label in CONFIGS:                      # SH -> B1_ocr -> B3full
    for qset in QSETS:                     # calib -> popevi -> popevi_t2
        run = logit_run_name(label, qset)
        if os.path.exists(f'{EXP_ROOT}/{run}/logits.json'):
            print(f'⏭ {label}/{qset}: đã có -> bỏ qua')
            continue
        cmd = (f'python -u -m src.eval.yesno_calib --score --data_file {Q_FILES[qset]} --run_name {run} '
               f'--drive_root {DRIVE_ROOT} --chunk {CHUNK} {OCR_ARGS}')
        if label in AD:
            cmd += f' --adapter {AD[label]}'
        run_cmd(cmd)
    print()
for label in CONFIGS:
    for qset in QSETS:
        m = json.load(open(f'{EXP_ROOT}/{logit_run_name(label, qset)}/meta.json', encoding='utf-8'))
        print(f"✔ {label:6s} {qset:10s} {m['num_samples']:5d} câu | {m['sec_per_sample']} s/câu | VRAM {m.get('peak_vram_gb')} GB")

## 5. Phân tích + phán quyết theo luật R-YB

`src.eval.yesno_calib --analyze` làm theo thứ tự:
1. **Kiểm hợp lệ**: đủ câu, đúng sha256 file câu hỏi, adapter trùng run greedy W12, OCR cùng thiết lập SH. Quyết định logit ở τ = 0 phải trùng greedy W12 ≥ 98%.
2. **AUC** (chỉ mô tả): còn phân biệt được câu có-thật / không-có hay không.
3. **τ*** của từng cấu hình, chọn trên calibration theo balanced accuracy.
4. **Luật R-YB**: SH với τ* so với B1-ocr greedy trên POPE-vi (CI95 bootstrap theo ảnh), cộng yes-ratio.
5. **Chỉ mô tả**: SH ở τ = 0, B1-ocr/B3-full với τ* của chính chúng, mẫu câu 2.

*Kỳ vọng: 6 bảng, dòng `➜ R-YB: **ĐẠT**` hoặc `**KHÔNG ĐẠT**`. File `yesno_calib_summary.json` + `.md` nằm trong `experiments/W13_yesno_calib_L4_seed42/`.*

In [ ]:
os.makedirs(OUT, exist_ok=True)
run_cmd(f'python -m src.eval.yesno_calib --analyze --pope {POPE_REPO} --questions_dir {Q_DIR} '
        f'--exp_root {EXP_ROOT} --out {OUT}')

## 6. Tự trả máy L4 khi chạy xong

Kết quả đã nằm trên Drive. Output phía trên vẫn hiện trong tab sau khi trả máy. Nếu notebook dừng sớm vì lỗi thì máy **không** tự trả: đọc lỗi xong, tự bấm **Runtime → Disconnect and delete runtime**.

*Kỳ vọng: `✅ Xong toàn bộ -> tự trả máy …`.*

In [ ]:
AUTO_RELEASE = True
if AUTO_RELEASE:
    print('✅ Xong toàn bộ -> tự trả máy để ngừng tính CU (output phía trên vẫn giữ trong tab).')
    from google.colab import runtime
    runtime.unassign()
else:
    print('AUTO_RELEASE = False -> nhớ tự Disconnect and delete runtime khi xong.')

## 7. ⏸ DỪNG

Gửi Claude output **mục 5**. Nếu lỡ đóng tab mất output thì không sao: kết quả nằm trong `experiments/W13_YN_*` và `W13_yesno_calib_L4_seed42/` trên Drive, Cursor chép và kiểm lại được.